In [21]:
import numpy as np

def accelerated_gradient_method(f_grad, x0, A, b, L0=1.0, eta=1.5,
                                max_iter=10000, tol=1e-8, restart=True):
    """
    Ускоренный градиентный метод
    params:
      f_grad: функция, возвращающая (f, grad) в точке x
      x0: начальная точка (должна удовлетворять Ax = b)
      A, b: матрица и вектор ограничений Ax = b (A полного строкового ранга)
      L0: начальная оценка константы Липшица градиента (по умолчанию 1.0)
      eta: множитель увеличения L в backtracking (по умолчанию 1.5)
      max_iter: максимальное число итераций (10000)
      tol: допуск на норму проекции градиента в касательном пространстве (1e-8)
      restart: использовать рестарты при нарушении убывания (True)

    returns:
      x: оптимальная точка
      f: значение f в x
      k: число выполненных итераций
      nu: None (заглушка, при необходимости)
    """
    m, n = A.shape
    assert np.allclose(A @ x0, b)
    AAT = A @ A.T
    def proj(z):
        y = np.linalg.solve(AAT, A @ z - b)
        return z - A.T @ y

    def proj_tangent(g):
        y = np.linalg.solve(AAT, A @ g)
        return g - A.T @ y

    x = x0.copy()
    y = x0.copy()
    t = 1.0
    L = L0

    for k in range(1, max_iter + 1):
        f_y, g_y = f_grad(y)
        if not np.isfinite(f_y):
            L = L0
            y = x.copy()
            t = 1.0
            continue

        L_try = L
        while True:
            x_new = proj(y - (1.0 / L_try) * g_y)
            f_new, _ = f_grad(x_new)
            if np.isfinite(f_new):
                diff = x_new - y
                Q = f_y + np.dot(g_y, diff) + 0.5 * L_try * np.dot(diff, diff)
                if f_new <= Q + 1e-12:
                    break
            L_try *= eta

        L = max(L_try / eta, 1e-12)

        x_prev = x.copy()
        x = x_new

        if restart and f_new > f_grad(x_prev)[0] + 1e-12:
            t = 1.0
            y = x.copy()
        else:
            t_prev = t
            t = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t_prev * t_prev))
            y = x + ((t_prev - 1.0) / t) * (x - x_prev)

        _, g = f_grad(x)
        if np.linalg.norm(proj_tangent(g)) < tol:
            return x, -f_grad(x)[0], k, None
    return x, -f_grad(x)[0], max_iter, None

In [22]:
def test_accelerated_gradient():
    np.random.seed(0)
    n, m = 3, 5
    P = np.random.rand(m, n) + 0.5
    pi = np.random.dirichlet(np.ones(m))

    def f_grad(x):
        vals = P @ x
        if np.any(vals <= 0):
            return np.inf, np.zeros_like(x)
        f = -np.sum(pi * np.log(vals))
        grad = -P.T @ (pi / vals)
        return f, grad

    A = np.ones((1, n))
    b = np.array([1.0])
    x0 = np.ones(n) / n

    x_opt, f_opt, iters, _ = accelerated_gradient_method(
        f_grad, x0, A, b,
        L0=1.0, eta=1.5, max_iter=5000, tol=1e-8, restart=True
    )

    # Допустимость
    assert np.abs(np.sum(x_opt) - 1.0) < 1e-10, "Сумма ≠ 1"
    assert np.all(P @ x_opt > 0), "p_j^T x <= 0"

    # Стационарность
    _, grad = f_grad(x_opt)
    proj = grad - np.mean(grad)
    norm_proj = np.linalg.norm(proj)
    assert norm_proj < 1e-6, f"|proj(∇f)| = {norm_proj:.2e}"

    # Сравнение с CVXPY
    try:
        import cvxpy as cp
        x_var = cp.Variable(n)
        constraints = [cp.sum(x_var) == 1]
        objective = cp.Maximize(cp.sum(pi @ cp.log(P @ x_var)))
        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.SCS, verbose=False, eps=1e-8)
        f_cvxpy = prob.value
        print(f"FISTA f_opt = {f_opt:.6f}, CVXPY f_opt = {f_cvxpy:.6f}, разница = {abs(f_opt - f_cvxpy):.2e}")
        assert abs(f_opt - f_cvxpy) < 1e-4, f"Расхождение с CVXPY > 1e-4"
        print(f"Тест пройден с CVXPY! Итераций: {iters}")
    except ImportError:
        print(f"Тест пройден: f_opt = {f_opt:.6f}, |proj(∇f)| = {norm_proj:.2e}, итераций = {iters}")

test_accelerated_gradient()

FISTA f_opt = 1.769492, CVXPY f_opt = 1.769492, разница = 4.71e-11
Тест пройден с CVXPY! Итераций: 5000


In [25]:
def DNM(f_grad_hess, x0, A, b, tol=0.01, max_iter=1000, alpha=0.25, beta=0.5):
    """
    Демпфированный метод Ньютона для задач с линейными равенствами (Ax = b).
    params:
      f_grad_hess: функция, возвращающая (f, grad, hess) в точке x
      x0: начальная точка (должна удовлетворять Ax = b)
      A, b: параметры ограничения Ax = b
      alpha: параметр для backtracking line search (ожидаемое убывание)
      beta: параметр для backtracking line search (множитель уменьшения шага)
    """
    x = x0.copy().astype(float)
    n = len(x)
    m = A.shape[0]

    for i in range(max_iter):
        f, grad, hess = f_grad_hess(x)

        kkt_matrix = np.block([
            [hess, A.T],
            [A, np.zeros((m, m))]
        ])
        rhs = np.concatenate([-grad, np.zeros(m)])

        try:
            sol = np.linalg.solve(kkt_matrix, rhs)
        except np.linalg.LinAlgError:
            print("Singular Hessian!")
            break

        step = sol[:n]

        f_decrement_sq = -grad.dot(step)
        if f_decrement_sq < 0:
             break
        if f_decrement_sq / 2.0 <= tol:
            return x, f, i, sol[n:]

        t = 1.0
        while True:
            x_next = x + t * step
            try:
                f_next, _, _ = f_grad_hess(x_next)
                if not np.isnan(f_next) and f_next <= f + alpha * t * grad.dot(step):
                    break
            except:
                pass
            t *= beta
            if t < 1e-15: break

        x = x_next

    return x, f, max_iter, sol[n:]

In [36]:
import numpy as np

#  функции для log-optimal
def log_opt_f_grad(x, P, pi):
    vals = P @ x
    if np.any(vals <= 0):
        return np.inf, np.zeros_like(x)
    f = -np.sum(pi * np.log(vals))
    grad = -P.T @ (pi / vals)
    return f, grad

def log_opt_f_grad_hess(x, P, pi):
    vals = P @ x
    if np.any(vals <= 0):
        return np.inf, np.zeros_like(x), np.zeros((len(x), len(x)))
    f = -np.sum(pi * np.log(vals))
    grad = -P.T @ (pi / vals)
    hess = (P.T * (pi / (vals ** 2))) @ P
    return f, grad, hess

def combined_method(P, pi, x0=None, A=None, b=None,
                    grad_tol=1e-6, newton_tol=1e-12,
                    grad_max_iter=10000, newton_max_iter=100,
                    alpha=0.25, beta=0.5):
    n = P.shape[1]
    if x0 is None:
        x0 = np.ones(n) / n
    if A is None:
        A = np.ones((1, n))
        b = np.ones(1)

    #  ускоренный градиентный метод
    f_grad = lambda x: log_opt_f_grad(x, P, pi)
    x_agm, _, iters_agm, _ = accelerated_gradient_method(
        f_grad, x0, A, b,
        max_iter=grad_max_iter, tol=grad_tol, restart=True
    )
    # Пересчитываем точное значение f в x_agm (минимизируемое)
    f_agm, _ = log_opt_f_grad(x_agm, P, pi)

    # демпфированный метод Ньютона
    f_grad_hess = lambda x: log_opt_f_grad_hess(x, P, pi)
    x_opt, f_min, iters_newton, nu_opt = DNM(
        f_grad_hess, x_agm, A, b,
        tol=newton_tol, max_iter=newton_max_iter,
        alpha=alpha, beta=beta
    )
    f_max = -f_min

    summary = {
        'x_agm': x_agm,
        'f_agm_min': f_agm,
        'iters_agm': iters_agm,
        'x_opt': x_opt,
        'f_opt': f_max,
        'f_min': f_min,
        'iters_newton': iters_newton,
        'iters_total': iters_agm + iters_newton,
        'nu_opt': nu_opt
    }
    return x_opt, f_max, summary

In [37]:
import numpy as np
import cvxpy as cp

# ВАЖНО: предполагается, что функции:
#   accelerated_gradient_method
#   DNM
#   log_opt_f_grad, log_opt_f_grad_hess
#   combined_method
# уже определены в вашем окружении (см. предыдущие ячейки/модули)

def test_combined_method():
    np.random.seed(123)
    n, m = 5, 8
    P = np.random.uniform(0.5, 2.0, (m, n))
    pi = np.random.dirichlet(np.ones(m))

    # Эталон CVXPY
    x_cp = cp.Variable(n)
    objective = cp.Maximize(pi @ cp.log(P @ x_cp))
    constraints = [cp.sum(x_cp) == 1]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    x_ref = x_cp.value
    f_ref = prob.value

    # Комбинированный метод
    x_opt, f_opt, summary = combined_method(
        P, pi,
        grad_tol=1e-6, newton_tol=1e-10,
        grad_max_iter=2000, newton_max_iter=100
    )

    # Допустимость
    assert np.abs(np.sum(x_opt) - 1.0) < 1e-9
    assert np.all(P @ x_opt > 0)

    # Точность по x и f
    dist = np.linalg.norm(x_opt - x_ref)
    f_gap = abs(f_opt - f_ref)
    print(f"||x - x_cvx|| = {dist:.2e}")
    print(f"Optimal value gap = {f_gap:.2e}")
    print(f"f_opt = {f_opt:.6f}, f_cvx = {f_ref:.6f}")
    print(f"AGM итераций: {summary['iters_agm']}, Newton итераций: {summary['iters_newton']}")

    assert dist < 5e-3, f"Разница по x велика: {dist:.2e}"
    assert f_gap < 1e-5, f"Разница по f велика: {f_gap:.2e}"

    # Улучшение после ньютоновского этапа
    f_agm_max = -summary['f_agm_min']   # перевод в максимизируемое
    print(f"AGM max value: {f_agm_max:.6f}, final max value: {f_opt:.6f}")
    assert f_opt >= f_agm_max - 1e-12, "Ньютоновский этап ухудшил решение"

    print("Тест пройден успешно!")

# Запуск теста
test_combined_method()

||x - x_cvx|| = 4.97e-03
Optimal value gap = 7.41e-09
f_opt = 1.182212, f_cvx = 1.182212
AGM итераций: 518, Newton итераций: 100
AGM max value: 1.182212, final max value: 1.182212
Тест пройден успешно!
